# Stage 5 - Song assembly & rendering

Generate MusicGen instrumental sections -> cross-fade stitch -> mix the vocal stem -> master (WAV + MP3).

GPU is only needed to *generate* instrumentals (reuses Stage 2). Stitch/mix/master are CPU.

In [ ]:
import sys, os
from pathlib import Path
REPO_DIR = '/kaggle/working/metalcore'
if not Path(REPO_DIR).exists():
    !git clone https://github.com/YOUR_USERNAME/metalcore.git {REPO_DIR}
sys.path.insert(0, REPO_DIR); os.chdir(REPO_DIR)
# Assembly deps. Also need MusicGen deps if generating instrumentals here.
!pip install -q -r requirements-assembly.txt
!pip install -q -r requirements-music.txt   # only if generating instrumentals in this notebook
!apt-get -qq install -y ffmpeg >/dev/null 2>&1 || true

In [ ]:
import glob
ADAPTER = sorted(glob.glob('/kaggle/working/outputs/music/checkpoints/step_*/adapter'))[-1]
VOCAL   = '/kaggle/working/outputs/vocals/song_vocal.wav'   # from Stage 4 (optional)
print('adapter:', ADAPTER)

In [ ]:
# Full song: generate instrumental sections, mix the vocal, master to WAV+MP3.
# Drop --vocal for an instrumental-only render.
!python -m inference.cli song \
    --config configs/assembly.yaml \
    --music-config configs/music_lora.yaml \
    --adapter {ADAPTER} \
    --vocal {VOCAL} \
    --output /kaggle/working/outputs/songs/track01

In [ ]:
import IPython.display as ipd
ipd.Audio('/kaggle/working/outputs/songs/track01.wav')

Edit the `sections:` list in `configs/assembly.yaml` to change the arrangement, prompts and lengths.
Utility subcommands: `inference.cli stitch` (cross-fade section WAVs) and `inference.cli master` (loudness + export).